# Customer Shopping Behaviour Analysis: Data Preparation (Python)

**Goal:** Clean and transform the raw customer shopping dataset so it is ready for analysis in SQL and visualisation in Power BI.

**Business question:** *How can the company leverage consumer shopping data to identify trends, improve customer engagement, and optimize marketing and product strategies?*

**Dataset:** `data/customer_shopping_behavior.csv` has 3,900 customers and 18 columns covering demographics, purchase details, shipping and payment preferences, and loyalty indicators.

### Steps in this notebook
1. Load the data
2. Explore the data
3. Handle missing values
4. Standardise column names
5. Feature engineering (`age_group`, `purchase_frequency_days`)
6. Remove redundant columns
7. Final check of the clean dataset
8. Load the clean data into PostgreSQL

## 1. Load the data

In [1]:
import pandas as pd

# Load the raw dataset from the repository's data folder
df = pd.read_csv('../data/customer_shopping_behavior.csv')

## 2. Explore the data
A first look at the rows, column data types, and summary statistics.

In [2]:
# Preview the first 5 rows
df.head()

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
0,1,55,Male,Blouse,Clothing,53,Kentucky,L,Gray,Winter,3.1,Yes,Express,Yes,Yes,14,Venmo,Fortnightly
1,2,19,Male,Sweater,Clothing,64,Maine,L,Maroon,Winter,3.1,Yes,Express,Yes,Yes,2,Cash,Fortnightly
2,3,50,Male,Jeans,Clothing,73,Massachusetts,S,Maroon,Spring,3.1,Yes,Free Shipping,Yes,Yes,23,Credit Card,Weekly
3,4,21,Male,Sandals,Footwear,90,Rhode Island,M,Maroon,Spring,3.5,Yes,Next Day Air,Yes,Yes,49,PayPal,Weekly
4,5,45,Male,Blouse,Clothing,49,Oregon,M,Turquoise,Spring,2.7,Yes,Free Shipping,Yes,Yes,31,PayPal,Annually


In [3]:
# Column names, data types, and non-null counts
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3900 entries, 0 to 3899
Data columns (total 18 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   Customer ID             3900 non-null   int64  
 1   Age                     3900 non-null   int64  
 2   Gender                  3900 non-null   object 
 3   Item Purchased          3900 non-null   object 
 4   Category                3900 non-null   object 
 5   Purchase Amount (USD)   3900 non-null   int64  
 6   Location                3900 non-null   object 
 7   Size                    3900 non-null   object 
 8   Color                   3900 non-null   object 
 9   Season                  3900 non-null   object 
 10  Review Rating           3863 non-null   float64
 11  Subscription Status     3900 non-null   object 
 12  Shipping Type           3900 non-null   object 
 13  Discount Applied        3900 non-null   object 
 14  Promo Code Used         3900 non-null   

In [4]:
# Summary statistics for both numeric and categorical columns
df.describe(include='all')

,Customer ID,Age,Gender,Item Purchased,Category,Purchase Amount (USD),Location,Size,Color,Season,Review Rating,Subscription Status,Shipping Type,Discount Applied,Promo Code Used,Previous Purchases,Payment Method,Frequency of Purchases
count,3900.000000,3900.000000,3900,3900,3900,3900.000000,3900,3900,3900,3900,3863.000000,3900,3900,3900,3900,3900.000000,3900,3900
unique,NaN,NaN,2,25,4,NaN,50,4,25,4,NaN,2,6,2,2,NaN,6,7
top,NaN,NaN,Male,Blouse,Clothing,NaN,Montana,M,Olive,Spring,NaN,No,Free Shipping,No,No,NaN,PayPal,Every 3 Months
freq,NaN,NaN,2652,171,1737,NaN,96,1755,177,999,NaN,2847,675,2223,2223,NaN,677,584
mean,1950.500000,44.068462,NaN,NaN,NaN,59.764359,NaN,NaN,NaN,NaN,3.750065,NaN,NaN,NaN,NaN,25.351538,NaN,NaN
std,1125.977353,15.207589,NaN,NaN,NaN,23.685392,NaN,NaN,NaN,NaN,0.716983,NaN,NaN,NaN,NaN,14.447125,NaN,NaN
min,1.000000,18.000000,NaN,NaN,NaN,20.000000,NaN,NaN,NaN,NaN,2.500000,NaN,NaN,NaN,NaN,1.000000,NaN,NaN
25%,975.750000,31.000000,NaN,NaN,NaN,39.000000,NaN,NaN,NaN,NaN,3.100000,NaN,NaN,NaN,NaN,13.000000,NaN,NaN
50%,1950.500000,44.000000,NaN,NaN,NaN,60.000000,NaN,NaN,NaN,NaN,3.800000,NaN,NaN,NaN,NaN,25.000000,NaN,NaN
75%,2925.250000,57.000000,NaN,NaN,NaN,81.000000,NaN,NaN,NaN,NaN,4.400000,NaN,NaN,NaN,NaN,38.000000,NaN,NaN


**Observations**
- The dataset has 3,900 rows and 18 columns, with one row per customer.
- `Review Rating` has only 3,863 non-null values, so **37 ratings are missing**.
- Every other column is complete.

## 3. Handle missing values

In [5]:
# Count missing values in each column
df.isnull().sum()

Customer ID                0
Age                        0
Gender                     0
Item Purchased             0
Category                   0
Purchase Amount (USD)      0
Location                   0
Size                       0
Color                      0
Season                     0
Review Rating             37
Subscription Status        0
Shipping Type              0
Discount Applied           0
Promo Code Used            0
Previous Purchases         0
Payment Method             0
Frequency of Purchases     0
dtype: int64

Only `Review Rating` has missing values. Instead of dropping those rows, each missing rating is filled with the **median rating of its product category**:
- Ratings can differ between categories, so a category-level value is more accurate than one overall value.
- The median is not pulled up or down by extreme ratings the way the mean is.

In [6]:
# Fill each missing review rating with the median rating of the same category.
# Example: a missing rating for a Clothing item gets the median rating of all Clothing items.
df['Review Rating'] = df.groupby('Category')['Review Rating'].transform(
    lambda x: x.fillna(x.median())
)

In [7]:
# Confirm that no missing values remain
df.isnull().sum()

Customer ID               0
Age                       0
Gender                    0
Item Purchased            0
Category                  0
Purchase Amount (USD)     0
Location                  0
Size                      0
Color                     0
Season                    0
Review Rating             0
Subscription Status       0
Shipping Type             0
Discount Applied          0
Promo Code Used           0
Previous Purchases        0
Payment Method            0
Frequency of Purchases    0
dtype: int64

## 4. Standardise column names
Column names are converted to **snake_case** (lowercase with underscores). This makes them easier to use in Python and SQL, because names with spaces or brackets would need quotes in every query.

In [8]:
# Lowercase all column names and replace spaces with underscores
df.columns = df.columns.str.lower()
df.columns = df.columns.str.replace(' ', '_')

# 'purchase_amount_(usd)' still contains brackets, so give it a cleaner name
df = df.rename(columns={'purchase_amount_(usd)': 'purchase_amount'})

df.columns

Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'promo_code_used', 'previous_purchases',
       'payment_method', 'frequency_of_purchases'],
      dtype='object')

## 5. Feature engineering

### 5.1 Age group
Customers are split into 4 age groups of roughly equal size using quartiles (`pd.qcut`). Groups make it easier to compare shopping behaviour across life stages than individual ages do.

In [9]:
# Split age into 4 equal-sized groups (quartiles) and label them
labels = ['Young Adult', 'Adult', 'Middle-aged', 'Senior']
df['age_group'] = pd.qcut(df['age'], q=4, labels=labels)

# Compare the new column with the original age
df[['age', 'age_group']].head(10)

,age,age_group
0,55,Middle-aged
1,19,Young Adult
2,50,Middle-aged
3,21,Young Adult
4,45,Middle-aged
5,46,Middle-aged
6,63,Senior
7,27,Young Adult
8,26,Young Adult
9,57,Middle-aged


In [10]:
# Age range and number of customers in each group
df.groupby('age_group', observed=True)['age'].agg(['min', 'max', 'count'])

,min,max,count
age_group,,,
Young Adult,18,31,1028
Adult,32,44,942
Middle-aged,45,57,986
Senior,58,70,944


### 5.2 Purchase frequency in days
`frequency_of_purchases` is stored as text ('Weekly', 'Quarterly', ...). It is converted into the approximate **number of days between purchases** so it can be used in calculations.

Some labels mean the same thing, so they map to the same number:
- 'Fortnightly' and 'Bi-Weekly' both become 14 days.
- 'Quarterly' and 'Every 3 Months' both become 90 days.

In [11]:
# Approximate number of days between purchases for each frequency label
frequency_mapping = {
    'Weekly': 7,
    'Fortnightly': 14,
    'Bi-Weekly': 14,        # same as Fortnightly
    'Monthly': 30,
    'Quarterly': 90,
    'Every 3 Months': 90,   # same as Quarterly
    'Annually': 365
}

df['purchase_frequency_days'] = df['frequency_of_purchases'].map(frequency_mapping)

# Compare the new column with the original labels
df[['frequency_of_purchases', 'purchase_frequency_days']].head(10)

,frequency_of_purchases,purchase_frequency_days
0,Fortnightly,14
1,Fortnightly,14
2,Weekly,7
3,Weekly,7
4,Annually,365
5,Weekly,7
6,Quarterly,90
7,Weekly,7
8,Annually,365
9,Quarterly,90


## 6. Remove redundant columns
`discount_applied` and `promo_code_used` look identical. Check that they match before dropping one of them.

In [12]:
# Quick side-by-side look at the two columns
df[['discount_applied', 'promo_code_used']].head(10)

,discount_applied,promo_code_used
0,Yes,Yes
1,Yes,Yes
2,Yes,Yes
3,Yes,Yes
4,Yes,Yes
5,Yes,Yes
6,Yes,Yes
7,Yes,Yes
8,Yes,Yes
9,Yes,Yes


In [13]:
# True means the two columns match in every row
bool((df['discount_applied'] == df['promo_code_used']).all())

True

The two columns match in all 3,900 rows, so `promo_code_used` adds no new information. It is dropped and `discount_applied` is kept.

In [14]:
# Drop the duplicate column
df = df.drop('promo_code_used', axis=1)

## 7. Final check of the clean dataset

In [15]:
# Size and columns of the cleaned dataset
print(f"Rows: {df.shape[0]}, Columns: {df.shape[1]}")
df.columns

Rows: 3900, Columns: 19


Index(['customer_id', 'age', 'gender', 'item_purchased', 'category',
       'purchase_amount', 'location', 'size', 'color', 'season',
       'review_rating', 'subscription_status', 'shipping_type',
       'discount_applied', 'previous_purchases', 'payment_method',
       'frequency_of_purchases', 'age_group', 'purchase_frequency_days'],
      dtype='object')

## 8. Load the clean data into PostgreSQL
The cleaned DataFrame is written to a PostgreSQL table called `customer` so it can be analysed with SQL.

**Before running this section:**
1. Install PostgreSQL and pgAdmin.
2. In pgAdmin, create a database named `customer_behaviour`.
3. Replace the password placeholder below with your own password. Do not commit your real password to GitHub.

In [16]:
# Libraries needed to connect Python to PostgreSQL (only needs to be done once).
# Uncomment the line below and run it if they are not installed yet.
# !pip install psycopg2-binary sqlalchemy

In [17]:
from sqlalchemy import create_engine
from urllib.parse import quote_plus

# Step 1: Connection details (replace with your own)
username = "postgres"             # default PostgreSQL user
password = "your_password"        # password set during PostgreSQL installation
host = "localhost"                # PostgreSQL is running on this computer
port = "5432"                     # default PostgreSQL port
database = "customer_behaviour"   # database created in pgAdmin

# quote_plus() safely encodes special characters in the password (such as @, #, %)
engine = create_engine(
    f"postgresql+psycopg2://{username}:{quote_plus(password)}@{host}:{port}/{database}"
)

# Step 2: Write the DataFrame to a table
# if_exists="replace" drops and recreates the table every time this cell runs
table_name = "customer"
df.to_sql(table_name, engine, if_exists="replace", index=False)

print(f"Data successfully loaded into table '{table_name}' in database '{database}'.")

Data successfully loaded into table 'customer' in database 'customer_behaviour'.


In [18]:
# Confirm the load by reading the row count back from PostgreSQL
pd.read_sql(f"SELECT COUNT(*) AS total_rows FROM {table_name}", engine)

,total_rows
0,3900


---
**Next step:** The `customer` table is now ready for SQL analysis of customer segments, loyalty, and purchase drivers.